# Notebook 12 — Climate Merge with Wind Exposure (HURDAT2)

**Purpose**: For each Florida ZIP3, compute hurricane wind exposure over the loan performance window (2017-2024) using HURDAT2 track data + SimpleMaps ZIP centroids. Merge these exposure features into the model-ready loan table.

**Inputs**
- `hurdat2_fl_window.parquet` (from Notebook 11)
- `uszips.csv` (SimpleMaps free download, place in `data/raw/`)
- `fl_static_features.parquet` (from Notebook 07)

**Output**
- `fl_model_ready.parquet` (**overwrites** the FEMA-based version from Notebook 08)

**Method — wind exposure at ZIP3 level**

For each (ZIP3, storm) pair:
1. Compute distance from the ZIP3 centroid to every 6-hourly track point of the storm
2. Take the **minimum distance** (closest approach) and the **wind speed at that closest approach**
3. Apply an exponential decay to get the effective wind exposure at the ZIP3:

    `exposure_wind_kt = wind_at_closest × exp(-min_distance_km / DECAY_SCALE)`

where `DECAY_SCALE = 100 km` produces a realistic profile:
- 0 km: 100% of peak wind
- 100 km: 37% of peak
- 200 km: 14% of peak
- 300 km: 5% of peak

This is an approximation. Real hurricane wind fields depend on radius of max winds, environmental pressure, storm asymmetry — parameters HURDAT2 doesn't fully provide. Our approach follows the "distance-to-track" convention used in econ climate finance papers (see Gete, Tsouderou & Wachter 2024). We document this as a modelling choice, not a claim of physical fidelity.

**Loan-level features produced**

- `max_wind_kt`: max exposure_wind across all storms in the window
- `n_storms_wind_gt34`: number of storms delivering >34 kt (tropical storm threshold)
- `n_storms_wind_gt64`: number of storms delivering >64 kt (hurricane threshold)
- `cumulative_wind_kt`: sum of exposure_wind across all storms (a "total wind energy" proxy)
- `min_dist_to_any_storm_km`: distance to the closest hurricane track point in the window


In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW_DIR  = Path("your/data/path/here")
DATA_DIR = Path("your/data/path/here")

IN_STORMS   = DATA_DIR / "hurdat2_fl_window.parquet"
IN_USZIPS   = RAW_DIR / "uszips.csv"
IN_STATIC   = DATA_DIR / "fl_static_features.parquet"
OUT_MDL     = DATA_DIR / "fl_model_ready.parquet"

assert IN_STORMS.exists(), f"Missing: {IN_STORMS} — rerun Notebook 11"
assert IN_USZIPS.exists(), f"Missing: {IN_USZIPS} — download from simplemaps.com"
assert IN_STATIC.exists(), f"Missing: {IN_STATIC}"


## 1. Load inputs

In [3]:
storms = pd.read_parquet(IN_STORMS)
uszips = pd.read_csv(IN_USZIPS, low_memory=False)
static = pd.read_parquet(IN_STATIC)

print(f"HURDAT2 FL storms: {storms['storm_id'].nunique()} storms, "
      f"{len(storms):,} track records")
print(f"US ZIPs         : {len(uszips):,} rows")
print(f"Static features : {len(static):,} loans")
print(f"\nuszips columns: {uszips.columns.tolist()}")


HURDAT2 FL storms: 53 storms, 1,602 track records
US ZIPs         : 33,782 rows
Static features : 19,625 loans

uszips columns: ['zip', 'lat', 'lng', 'city', 'state_id', 'state_name', 'zcta', 'parent_zcta', 'population', 'density', 'county_fips', 'county_name', 'county_weights', 'county_names_all', 'county_fips_all', 'imprecise', 'military', 'timezone']


## 2. Build FL ZIP3 centroids from SimpleMaps

SimpleMaps' `uszips.csv` gives one row per 5-digit ZIP with `lat`, `lng`, and (if available) `population`. We collapse to 3-digit ZIP3 using population-weighted average when available.


In [4]:
fl_zips = uszips[uszips['state_id'] == 'FL'].copy()
fl_zips['ZIP3'] = fl_zips['zip'].astype(str).str.zfill(5).str[:3]

print(f"FL 5-digit ZIPs: {len(fl_zips):,}")
print(f"FL 3-digit ZIP3s: {fl_zips['ZIP3'].nunique()}")

# Population-weighted centroid if population available, else unweighted
has_pop = 'population' in fl_zips.columns and fl_zips['population'].sum() > 0

def zip3_centroid(g):
    if has_pop and g['population'].sum() > 0:
        w = g['population'].astype(float)
        return pd.Series({
            'lat': np.average(g['lat'], weights=w),
            'lon': np.average(g['lng'], weights=w),
            'total_pop': w.sum(),
        })
    else:
        return pd.Series({
            'lat': g['lat'].mean(),
            'lon': g['lng'].mean(),
            'total_pop': np.nan,
        })


fl_zip3 = (fl_zips.groupby('ZIP3').apply(zip3_centroid).reset_index())
print(f"\nFL ZIP3 centroids computed (weighted by population: {has_pop})")
print(fl_zip3.head())


FL 5-digit ZIPs: 1,011
FL 3-digit ZIP3s: 25

FL ZIP3 centroids computed (weighted by population: True)
  ZIP3        lat        lon  total_pop
0  320  30.136253 -81.887687   746309.0
1  321  29.226048 -81.341765   685544.0
2  322  30.282425 -81.623628  1095658.0
3  323  30.416951 -84.257566   449162.0
4  324  30.413362 -85.665869   379719.0


C:\Users\30695\AppData\Local\Temp\ipykernel_33588\615743938.py:26: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  fl_zip3 = (fl_zips.groupby('ZIP3').apply(zip3_centroid).reset_index())


## 3. Compute ZIP3 × storm exposure

In [5]:
def haversine_km(lat1, lon1, lat2, lon2):
    """Vectorised great-circle distance in km."""
    R = 6371.0
    lat1r, lat2r = np.radians(lat1), np.radians(lat2)
    dlat = np.radians(lat2 - lat1)
    dlon = np.radians(lon2 - lon1)
    a = np.sin(dlat/2)**2 + np.cos(lat1r) * np.cos(lat2r) * np.sin(dlon/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))


DECAY_SCALE_KM = 100.0   # exposure_wind = wind_at_closest * exp(-dist / DECAY_SCALE_KM)

exposure_rows = []
for storm_id, g in storms.groupby('storm_id'):
    storm_lats  = g['lat'].values
    storm_lons  = g['lon'].values
    storm_winds = g['wind_kt'].values
    storm_year  = int(g['year'].iloc[0])
    storm_name  = g['name'].iloc[0]

    for _, z in fl_zip3.iterrows():
        d = haversine_km(z['lat'], z['lon'], storm_lats, storm_lons)
        i = int(np.nanargmin(d))
        min_d = float(d[i])
        wind_close = float(storm_winds[i]) if not np.isnan(storm_winds[i]) else 0.0
        exposure = wind_close * np.exp(-min_d / DECAY_SCALE_KM)

        exposure_rows.append({
            'ZIP3':             z['ZIP3'],
            'storm_id':         storm_id,
            'storm_name':       storm_name,
            'storm_year':       storm_year,
            'min_dist_km':      min_d,
            'wind_at_closest':  wind_close,
            'exposure_wind_kt': exposure,
        })

exposure = pd.DataFrame(exposure_rows)
print(f"Exposure rows (ZIP3 × storm): {len(exposure):,}")
print(f"ZIP3s: {exposure['ZIP3'].nunique()}, storms: {exposure['storm_id'].nunique()}")
print(f"\nExposure summary:")
print(exposure[['min_dist_km', 'wind_at_closest', 'exposure_wind_kt']].describe().round(2))


Exposure rows (ZIP3 × storm): 1,325
ZIP3s: 25, storms: 53

Exposure summary:
       min_dist_km  wind_at_closest  exposure_wind_kt
count      1325.00          1325.00           1325.00
mean        528.99            45.27              3.91
std         296.41            23.27              9.08
min           8.81            15.00              0.00
25%         281.04            30.00              0.02
50%         515.36            40.00              0.21
75%         765.72            55.00              2.61
max        1371.91           140.00            103.69


### Quick visual check — for each storm, which ZIP3s got hit hardest?

In [6]:
top_by_storm = (exposure
                .sort_values(['storm_id', 'exposure_wind_kt'], ascending=[True, False])
                .groupby('storm_id')
                .head(3))
print("Top 3 most-exposed ZIP3s per storm:")
print(top_by_storm[['storm_name', 'storm_year', 'ZIP3', 'min_dist_km', 
                    'wind_at_closest', 'exposure_wind_kt']].to_string(index=False))


Top 3 most-exposed ZIP3s per storm:
storm_name  storm_year ZIP3  min_dist_km  wind_at_closest  exposure_wind_kt
   ALBERTO        2018  324    34.448812             40.0         28.343319
   ALBERTO        2018  325    95.035291             30.0         11.598137
   ALBERTO        2018  323   167.686162             40.0          7.478392
    ARTHUR        2020  349   185.042768             30.0          4.715098
    ARTHUR        2020  329   190.775984             30.0          4.452374
    ARTHUR        2020  334   211.491010             30.0          3.619324
      ALEX        2022  349     8.810122             40.0         36.626728
      ALEX        2022  341    31.926066             35.0         25.434014
      ALEX        2022  339    53.672642             35.0         20.463067
     BARRY        2019  324   227.566063             25.0          2.568226
     BARRY        2019  325   231.438826             25.0          2.470666
     BARRY        2019  323   304.239553            

## 4. Aggregate ZIP3 exposure across all storms in the window

In [7]:
zip3_features = (exposure.groupby('ZIP3').agg(
    max_wind_kt=('exposure_wind_kt', 'max'),
    cumulative_wind_kt=('exposure_wind_kt', 'sum'),
    min_dist_to_any_storm_km=('min_dist_km', 'min'),
    n_storms_wind_gt34=('exposure_wind_kt', lambda s: (s > 34).sum()),
    n_storms_wind_gt64=('exposure_wind_kt', lambda s: (s > 64).sum()),
).reset_index())

print(f"ZIP3 features table: {len(zip3_features)} rows")
print(zip3_features.describe().round(2))
print(f"\nZIP3s with any hurricane-strength exposure (>64 kt): "
      f"{(zip3_features['n_storms_wind_gt64'] > 0).sum()}")


ZIP3 features table: 25 rows
       max_wind_kt  cumulative_wind_kt  min_dist_to_any_storm_km  \
count        25.00               25.00                     25.00   
mean         41.58              207.44                     44.43   
std          22.05               42.70                     25.55   
min          16.48              143.24                      8.81   
25%          25.17              177.71                     29.49   
50%          38.47              205.95                     38.59   
75%          47.43              233.33                     60.02   
max         103.69              302.36                    116.87   

       n_storms_wind_gt34  n_storms_wind_gt64  
count               25.00               25.00  
mean                 0.88                0.12  
std                  0.83                0.33  
min                  0.00                0.00  
25%                  0.00                0.00  
50%                  1.00                0.00  
75%                  2

## 5. Merge ZIP3 features onto the loan-level static table

In [8]:
model_df = static.merge(zip3_features, on='ZIP3', how='left')

n_missing = model_df['max_wind_kt'].isna().sum()
print(f"Loans with unmatched ZIP3 (no HURDAT2 features): {n_missing:,}")

# Fill missing with zeros (assumption: unmatched ZIP3s had no significant storm exposure)
climate_cols = ['max_wind_kt', 'cumulative_wind_kt', 
                'n_storms_wind_gt34', 'n_storms_wind_gt64']
for c in climate_cols:
    model_df[c] = model_df[c].fillna(0)

# min_dist: unmatched -> a large number (far away)
model_df['min_dist_to_any_storm_km'] = model_df['min_dist_to_any_storm_km'].fillna(9999)

print(f"\nFinal model-ready shape: {model_df.shape}")


Loans with unmatched ZIP3 (no HURDAT2 features): 0

Final model-ready shape: (19625, 29)


## 6. Sanity check — loan-level climate feature distribution

In [9]:
print("Loan-level climate feature distribution:")
print(model_df[climate_cols + ['min_dist_to_any_storm_km']].describe().round(2))

print("\nDefault rate by max_wind_kt quintile:")
model_df['wind_quintile'] = pd.qcut(model_df['max_wind_kt'], q=5, 
                                     duplicates='drop', labels=False)
xt = model_df.groupby('wind_quintile').agg(
    n=('default_180dpd', 'size'),
    default_rate=('default_180dpd', 'mean'),
    mean_max_wind=('max_wind_kt', 'mean'),
).round(4)
print(xt)

# Cleanup
model_df = model_df.drop(columns=['wind_quintile'])


Loan-level climate feature distribution:
       max_wind_kt  cumulative_wind_kt  n_storms_wind_gt34  \
count     19625.00            19625.00            19625.00   
mean         39.53              203.01                0.73   
std          21.16               41.33                0.77   
min          16.48              143.24                0.00   
25%          25.17              177.71                0.00   
50%          34.20              191.83                1.00   
75%          47.43              231.93                1.00   
max         103.69              302.36                2.00   

       n_storms_wind_gt64  min_dist_to_any_storm_km  
count            19625.00                  19625.00  
mean                 0.09                     47.58  
std                  0.29                     25.55  
min                  0.00                      8.81  
25%                  0.00                     29.49  
50%                  0.00                     39.57  
75%                  0

## 7. Save (overwrites the FEMA-based fl_model_ready.parquet)

In [10]:
model_df.to_parquet(OUT_MDL, index=False)
print(f"Saved: {OUT_MDL}")
print(f"Shape: {model_df.shape}")
print(f"New climate columns: {climate_cols + ['min_dist_to_any_storm_km']}")


Saved: E:\Financial Mathsmatics Master\Dissertation\datasets\Data Processed\updated data\fl_model_ready.parquet
Shape: (19625, 29)
New climate columns: ['max_wind_kt', 'cumulative_wind_kt', 'n_storms_wind_gt34', 'n_storms_wind_gt64', 'min_dist_to_any_storm_km']


In [2]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path("E:/Financial Mathsmatics Master/Dissertation/datasets/Data Processed/updated data/")   # 改成你的路径

model_df = pd.read_parquet(DATA_DIR / "fl_model_ready.parquet")

climate_cols = [
    'max_wind_kt',
    'n_storms_wind_gt34',
    'n_storms_wind_gt64',
    'cumulative_wind_kt',
    'min_dist_to_any_storm_km',
]

print("Table 3.X: Climate Variable Summary Statistics")
print("=" * 60)
print(model_df[climate_cols].describe().round(2))

Table 3.X: Climate Variable Summary Statistics
       max_wind_kt  n_storms_wind_gt34  n_storms_wind_gt64  \
count     19625.00            19625.00            19625.00   
mean         39.53                0.73                0.09   
std          21.16                0.77                0.29   
min          16.48                0.00                0.00   
25%          25.17                0.00                0.00   
50%          34.20                1.00                0.00   
75%          47.43                1.00                0.00   
max         103.69                2.00                1.00   

       cumulative_wind_kt  min_dist_to_any_storm_km  
count            19625.00                  19625.00  
mean               203.01                     47.58  
std                 41.33                     25.55  
min                143.24                      8.81  
25%                177.71                     29.49  
50%                191.83                     39.57  
75%             

## 8. What to expect and how to interpret next

After running this notebook, you have a `fl_model_ready.parquet` with **5 new climate variables** replacing the two FEMA-based ones.

**Key expectations before rerunning Notebook 10 (M2 logit):**

1. **max_wind_kt should have real variation** — expect min close to 0 (interior northern FL ZIP3s away from storms), max in the 80-120 kt range (South FL ZIP3s hit by Irma or Ian).

2. **Default rate quintile check** — the quintile table at step 6 should show a **monotonic increase** in default rate from low-wind to high-wind quintiles. If it does, that's raw evidence of a climate signal (before controls).

3. **If the quintile pattern is monotonic but weak** — the LR test in M2 might still be significant because logistic regression pools across quintiles. Small monotonic effects can still yield p < 0.05 with n = 19,625.

4. **If the quintile pattern is flat or non-monotonic** — HURDAT2 exposure also doesn't identify a signal in this sample. That would be a substantive finding (GSE market absorbs climate risk), not a data problem.

**Next steps after this notebook**
- Update Notebook 10 Section 2 numeric_features to swap in the new climate variables
- Rerun M2 logit
- Compare M1 vs M2 metrics
